# Sentiment Classification Pipeline

## Objective

Classify cleaned customer support ticket descriptions using a pretrained Hugging Face sentiment analysis pipeline.

The workflow includes:

- Loading the cleaned ticket dataset
- Testing the pretrained pipeline on a small sample
- Running inference across all ticket descriptions
- Validating predictions against a manually labeled sample
- Documenting observed failure modes
- Exporting ticket-level sentiment results for later SQL integration

In [48]:
# Import project path utilities, data handling tools, and the pretrained NLP pipeline.
from pathlib import Path

import pandas as pd
from transformers import pipeline

In [49]:
# Define the project root relative to the notebook location.
PROJECT_ROOT = Path.cwd().parent

# Point to the cleaned dataset produced during preprocessing.
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_support_tickets_cleaned.csv"
)

# Load the cleaned ticket dataset for sentiment analysis.
df = pd.read_csv(DATA_PATH)

# Confirm the expected dataset dimensions before proceeding.
print(f"Dataset shape: {df.shape}")

df.head()

Dataset shape: (8469, 20)


,Ticket ID,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating,Cleaned Ticket Description,Has First Response,Is Resolved,Has CSAT,is_dissatisfied
0,1,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,"I'm having an issue with the {product_purchased}. Please assist. Your billing zip code is: 71701. We appreciate that you have requested a website address. Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.",Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN,"I'm having an issue with the GoPro Hero. Please assist. Your billing zip code is: 71701. We appreciate that you have requested a website address. Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.",1,0,0,NaN
1,2,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,"I'm having an issue with the {product_purchased}. Please assist. If you need to change an existing product. I'm having an issue with the {product_purchased}. Please assist. If The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.",Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN,"I'm having an issue with the LG Smart TV. Please assist. If you need to change an existing product. I'm having an issue with the LG Smart TV. Please assist. If The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.",1,0,0,NaN
2,3,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,"I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was working fine until yesterday, but now it doesn't respond. 1.8.3 I really I'm using the original charger that came with my {product_purchased}, but it's not charging properly.",Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0,"I'm facing a problem with my Dell XPS. The Dell XPS is not turning on. It was working fine until yesterday, but now it doesn't respond. 1.8.3 I really I'm using the original charger that came with my Dell XPS, but it's not charging properly.",1,1,1,0.0
3,4,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,"I'm having an issue with the {product_purchased}. Please assist. If you have a problem you're interested in and I'd love to see this happen, please check out the Feedback. I've already contacted customer support multiple times, but the issue remains unresolved.",Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0,"I'm having an issue with the Microsoft Office. Please assist. If you have a problem you're interested in and I'd love to see this happen, please check out the Feedback. I've already contacted customer support multiple times, but the issue remains unresolved.",1,1,1,0.0
4,5,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchased}. Please assist. Note: The seller is not responsible for any damages arising out of the delivery of the battleground game. Please have the game in good condition and shipped to you I've noticed a sudden decrease in battery life on my {product_purchased}. It used to last much longer.,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0,I'm having an issue with the Autodesk AutoCAD. Please assist. Note: The seller is not responsible for any damages arising out of the delivery of the battleground game. Please have the game in good condition and shipped to you I've noticed a sudden decrease in battery life on my Autodesk AutoCAD. It used to last much longer.,1,1,

## 1. Text Input Inspection

The sentiment pipeline will use `Cleaned Ticket Description`, which was standardized during preprocessing and has product placeholders replaced with their corresponding product values.

Before running the model, verify that the column is complete and inspect representative ticket descriptions.

In [50]:
# Check for missing or empty ticket descriptions before model inference.
text_col = "Cleaned Ticket Description"

missing_descriptions = df[text_col].isna().sum()
empty_descriptions = (
    df[text_col]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(f"Missing descriptions: {missing_descriptions}")
print(f"Empty descriptions: {empty_descriptions}")

Missing descriptions: 0
Empty descriptions: 0


In [51]:
# Display a small sample of cleaned ticket descriptions for manual inspection.
df[
    [
        "Ticket ID",
        "Ticket Type",
        "Ticket Subject",
        "Cleaned Ticket Description",
    ]
].sample(10, random_state=42)

,Ticket ID,Ticket Type,Ticket Subject,Cleaned Ticket Description
4830,4831,Refund request,Product setup,"I'm having an issue with the Roomba Robot Vacuum. Please assist. I'm using xda-developer for something different. If there are issues with the Roomba Robot Vacuum it's likely you are not using the I've tried clearing the cache and data for the Roomba Robot Vacuum app, but the issue persists."
7075,7076,Product inquiry,Battery life,"I'm having trouble connecting my Roomba Robot Vacuum to my home Wi-Fi network. It doesn't detect any networks, although other devices are connecting fine. What can be done to resolve this issue? I will refer to this issue I've checked for any available software updates for my Roomba Robot Vacuum, but there are none."
4715,4716,Billing inquiry,Refund request,I'm having an issue with the Philips Hue Lights. Please assist. Please give credit to: @joeyclay I'm concerned about the security of my Philips Hue Lights and would like to ensure that my data is safe.
2022,2023,Billing inquiry,Peripheral compatibility,I'm having an issue with the LG OLED. Please assist. 4. Check and compare product pricing You will see that prices are based on the current prices on your credit card. If you are a resident of I've noticed a peculiar error message popping up on my LG OLED screen. It says ' '. What does it mean?
676,677,Refund request,Peripheral compatibility,I'm having an issue with the Roomba Robot Vacuum. Please assist. I would like my price to rise so that I can return it. Please notify me if you do not want their refund. Please do I've noticed a sudden decrease in battery life on my Roomba Robot Vacuum. It used to last much longer.
2283,2284,Refund request,Refund request,I've encountered a data loss issue with my Sony PlayStation. All the files and documents seem to have disappeared. Can you guide me on how to retrieve them? I cannot verify this information though. It doesn't look very I've noticed that the issue occurs consistently when I use a specific feature or application on my Sony PlayStation.
5076,5077,Product inquiry,Software bug,I'm having an issue with the Sony 4K HDR TV. Please assist. I've noticed a peculiar error message popping up on my Sony 4K HDR TV screen. It says ' '. What does it mean?
2476,2477,Refund request,Installation support,I'm having an issue with the Xbox. Please assist. If you are having trouble with your package you need to contact customer service with the following: Website P.O. Box 300 I've noticed a sudden decrease in battery life on my Xbox. It used to last much longer.
6847,6848,Refund request,Hardware issue,"I'm encountering a software bug in the Microsoft Office. Whenever I try to perform a specific action, the application crashes. Are there any updates or fixes available? Will my account get frozen in order to save money? Is my password I've checked for software updates, and my Microsoft Office is already running the latest version."
511,512,Product inquiry,Battery life,"I'm having an issue with the Canon EOS. Please assist. Thanks, and thanks a lot, I've tried clearing the cache and data for the Canon EOS app, but the issue persists."


## 2. Pretrained Sentiment Model

A pretrained Hugging Face sentiment analysis pipeline is used to classify the emotional polarity of customer support ticket descriptions.

The workflow uses the `distilbert/distilbert-base-uncased-finetuned-sst-2-english` model, which produces binary `POSITIVE` and `NEGATIVE` sentiment labels.

The model is tested on a small reproducible sample before running inference across the full dataset.

In [52]:
# Initialize a pretrained sentiment model for binary sentiment classification.
MODEL_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

sentiment_classifier = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [53]:
# Select a reproducible sample of ticket descriptions for an initial model check.
sample_tickets = df[
    [
        "Ticket ID",
        "Cleaned Ticket Description",
    ]
].sample(10, random_state=42)

sample_texts = sample_tickets[
    "Cleaned Ticket Description"
].tolist()

In [54]:
# Run sentiment inference on the sample tickets.
sample_predictions = sentiment_classifier(sample_texts)

sample_predictions

[{'label': 'NEGATIVE', 'score': 0.9963129162788391},
 {'label': 'NEGATIVE', 'score': 0.999211311340332},
 {'label': 'POSITIVE', 'score': 0.9917927384376526},
 {'label': 'NEGATIVE', 'score': 0.9975979924201965},
 {'label': 'NEGATIVE', 'score': 0.9980171918869019},
 {'label': 'NEGATIVE', 'score': 0.9997377991676331},
 {'label': 'NEGATIVE', 'score': 0.9971539974212646},
 {'label': 'NEGATIVE', 'score': 0.9986587762832642},
 {'label': 'NEGATIVE', 'score': 0.9997127652168274},
 {'label': 'NEGATIVE', 'score': 0.9220516085624695}]

In [55]:
# Combine the sample tickets with their predicted sentiment labels and scores.
sample_results = sample_tickets.copy()

sample_results["sentiment_label"] = [
    prediction["label"]
    for prediction in sample_predictions
]

sample_results["sentiment_score"] = [
    prediction["score"]
    for prediction in sample_predictions
]

sample_results

,Ticket ID,Cleaned Ticket Description,sentiment_label,sentiment_score
4830,4831,"I'm having an issue with the Roomba Robot Vacuum. Please assist. I'm using xda-developer for something different. If there are issues with the Roomba Robot Vacuum it's likely you are not using the I've tried clearing the cache and data for the Roomba Robot Vacuum app, but the issue persists.",NEGATIVE,0.996313
7075,7076,"I'm having trouble connecting my Roomba Robot Vacuum to my home Wi-Fi network. It doesn't detect any networks, although other devices are connecting fine. What can be done to resolve this issue? I will refer to this issue I've checked for any available software updates for my Roomba Robot Vacuum, but there are none.",NEGATIVE,0.999211
4715,4716,I'm having an issue with the Philips Hue Lights. Please assist. Please give credit to: @joeyclay I'm concerned about the security of my Philips Hue Lights and would like to ensure that my data is safe.,POSITIVE,0.991793
2022,2023,I'm having an issue with the LG OLED. Please assist. 4. Check and compare product pricing You will see that prices are based on the current prices on your credit card. If you are a resident of I've noticed a peculiar error message popping up on my LG OLED screen. It says ' '. What does it mean?,NEGATIVE,0.997598
676,677,I'm having an issue with the Roomba Robot Vacuum. Please assist. I would like my price to rise so that I can return it. Please notify me if you do not want their refund. Please do I've noticed a sudden decrease in battery life on my Roomba Robot Vacuum. It used to last much longer.,NEGATIVE,0.998017
2283,2284,I've encountered a data loss issue with my Sony PlayStation. All the files and documents seem to have disappeared. Can you guide me on how to retrieve them? I cannot verify this information though. It doesn't look very I've noticed that the issue occurs consistently when I use a specific feature or application on my Sony PlayStation.,NEGATIVE,0.999738
5076,5077,I'm having an issue with the Sony 4K HDR TV. Please assist. I've noticed a peculiar error message popping up on my Sony 4K HDR TV screen. It says ' '. What does it mean?,NEGATIVE,0.997154
2476,2477,I'm having an issue with the Xbox. Please assist. If you are having trouble with your package you need to contact customer service with the following: Website P.O. Box 300 I've noticed a sudden decrease in battery life on my Xbox. It used to last much longer.,NEGATIVE,0.998659
6847,6848,"I'm encountering a software bug in the Microsoft Office. Whenever I try to perform a specific action, the application crashes. Are there any updates or fixes available? Will my account get frozen in order to save money? Is my password I've checked for software updates, and my Microsoft Office is already running the latest version.",NEGATIVE,0.999713
511,512,"I'm having an issue with the Canon EOS. Please assist. Thanks, and thanks a lot, I've tried clearing the cache and data for the Canon EOS app, but the issue persists.",NEGATIVE,0.922052


In [56]:
# Display full ticket descriptions to support manual inspection of sample predictions.
pd.set_option("display.max_colwidth", None)

sample_results

,Ticket ID,Cleaned Ticket Description,sentiment_label,sentiment_score
4830,4831,"I'm having an issue with the Roomba Robot Vacuum. Please assist. I'm using xda-developer for something different. If there are issues with the Roomba Robot Vacuum it's likely you are not using the I've tried clearing the cache and data for the Roomba Robot Vacuum app, but the issue persists.",NEGATIVE,0.996313
7075,7076,"I'm having trouble connecting my Roomba Robot Vacuum to my home Wi-Fi network. It doesn't detect any networks, although other devices are connecting fine. What can be done to resolve this issue? I will refer to this issue I've checked for any available software updates for my Roomba Robot Vacuum, but there are none.",NEGATIVE,0.999211
4715,4716,I'm having an issue with the Philips Hue Lights. Please assist. Please give credit to: @joeyclay I'm concerned about the security of my Philips Hue Lights and would like to ensure that my data is safe.,POSITIVE,0.991793
2022,2023,I'm having an issue with the LG OLED. Please assist. 4. Check and compare product pricing You will see that prices are based on the current prices on your credit card. If you are a resident of I've noticed a peculiar error message popping up on my LG OLED screen. It says ' '. What does it mean?,NEGATIVE,0.997598
676,677,I'm having an issue with the Roomba Robot Vacuum. Please assist. I would like my price to rise so that I can return it. Please notify me if you do not want their refund. Please do I've noticed a sudden decrease in battery life on my Roomba Robot Vacuum. It used to last much longer.,NEGATIVE,0.998017
2283,2284,I've encountered a data loss issue with my Sony PlayStation. All the files and documents seem to have disappeared. Can you guide me on how to retrieve them? I cannot verify this information though. It doesn't look very I've noticed that the issue occurs consistently when I use a specific feature or application on my Sony PlayStation.,NEGATIVE,0.999738
5076,5077,I'm having an issue with the Sony 4K HDR TV. Please assist. I've noticed a peculiar error message popping up on my Sony 4K HDR TV screen. It says ' '. What does it mean?,NEGATIVE,0.997154
2476,2477,I'm having an issue with the Xbox. Please assist. If you are having trouble with your package you need to contact customer service with the following: Website P.O. Box 300 I've noticed a sudden decrease in battery life on my Xbox. It used to last much longer.,NEGATIVE,0.998659
6847,6848,"I'm encountering a software bug in the Microsoft Office. Whenever I try to perform a specific action, the application crashes. Are there any updates or fixes available? Will my account get frozen in order to save money? Is my password I've checked for software updates, and my Microsoft Office is already running the latest version.",NEGATIVE,0.999713
511,512,"I'm having an issue with the Canon EOS. Please assist. Thanks, and thanks a lot, I've tried clearing the cache and data for the Canon EOS app, but the issue persists.",NEGATIVE,0.922052


## 3. Full-Dataset Sentiment Inference

After confirming that the pretrained pipeline runs successfully on a sample, sentiment predictions are generated for every cleaned ticket description.

Each prediction contains:

- A binary sentiment label
- A model confidence score

The resulting fields will be retained at the ticket level for later analytical integration.

In [57]:
# Extract all cleaned ticket descriptions for batch sentiment inference.
all_texts = df[text_col].tolist()

# Run sentiment inference across the complete dataset.
all_predictions = sentiment_classifier(
    all_texts,
    batch_size=32,
    truncation=True,
)

In [58]:
# Confirm that every ticket received a sentiment prediction.
print(f"Input records: {len(all_texts)}")
print(f"Predictions generated: {len(all_predictions)}")

Input records: 8469
Predictions generated: 8469


## 4. Sentiment Results

The model predictions are combined with ticket identifiers to create a ticket-level sentiment dataset.

The output retains the predicted sentiment label and model confidence score for each ticket.

In [59]:
# Create a ticket-level results table from the full set of model predictions.
sentiment_results = df[["Ticket ID"]].copy()

sentiment_results["sentiment_label"] = [
    prediction["label"]
    for prediction in all_predictions
]

sentiment_results["sentiment_score"] = [
    prediction["score"]
    for prediction in all_predictions
]

sentiment_results.head()

,Ticket ID,sentiment_label,sentiment_score
0,1,NEGATIVE,0.994507
1,2,NEGATIVE,0.948731
2,3,NEGATIVE,0.999365
3,4,NEGATIVE,0.967981
4,5,NEGATIVE,0.997396


In [60]:
# Verify that the sentiment results contain one complete prediction per ticket.
print(f"Results shape: {sentiment_results.shape}")
print(f"Unique Ticket IDs: {sentiment_results['Ticket ID'].nunique()}")
print(f"Missing sentiment labels: {sentiment_results['sentiment_label'].isna().sum()}")
print(f"Missing sentiment scores: {sentiment_results['sentiment_score'].isna().sum()}")

Results shape: (8469, 3)
Unique Ticket IDs: 8469
Missing sentiment labels: 0
Missing sentiment scores: 0


In [61]:
# Examine the distribution of predicted sentiment labels.
sentiment_results["sentiment_label"].value_counts()

sentiment_label
NEGATIVE    7733
POSITIVE     736
Name: count, dtype: int64

In [62]:
# Summarize the model confidence scores across all predictions.
sentiment_results["sentiment_score"].describe()

count    8469.000000
mean        0.977301
std         0.070449
min         0.501143
25%         0.992853
50%         0.998227
75%         0.999480
max         0.999821
Name: sentiment_score, dtype: float64

## 5. Prediction Distribution

The overall distribution of predicted sentiment labels is reviewed to identify potential class imbalance and assess whether the pretrained model's behavior appears reasonable for customer support text.

Model confidence is interpreted separately from prediction correctness, as high-confidence predictions may still be affected by domain mismatch or noisy ticket content.

In [63]:
# Calculate sentiment counts and percentages across the full dataset.
sentiment_distribution = (
    sentiment_results["sentiment_label"]
    .value_counts()
    .rename_axis("sentiment_label")
    .reset_index(name="ticket_count")
)

sentiment_distribution["percentage"] = (
    sentiment_distribution["ticket_count"]
    / len(sentiment_results)
    * 100
).round(2)

sentiment_distribution

,sentiment_label,ticket_count,percentage
0,NEGATIVE,7733,91.31
1,POSITIVE,736,8.69


## 6. Manual Validation Sample

A balanced, reproducible sample is selected from the predicted sentiment classes for manual review.

The sample is used to compare model predictions with manually assigned sentiment labels and to identify common failure modes. Because the full prediction set is heavily skewed toward `NEGATIVE`, sampling equally from both predicted classes provides better coverage of the less frequent `POSITIVE` predictions.

This validation is intended as a qualitative model check rather than a statistically representative evaluation of overall model accuracy.

In [64]:
# Select a balanced and reproducible sample for manual validation.
validation_sample = (
    sentiment_results
    .groupby("sentiment_label", group_keys=False)
    .sample(n=10, random_state=42)
    .sort_values("Ticket ID")
    .reset_index(drop=True)
)

# Attach the corresponding ticket descriptions for manual review.
validation_sample = validation_sample.merge(
    df[["Ticket ID", "Cleaned Ticket Description"]],
    on="Ticket ID",
    how="left",
)

validation_sample[
    [
        "Ticket ID",
        "Cleaned Ticket Description",
        "sentiment_label",
        "sentiment_score",
    ]
]

,Ticket ID,Cleaned Ticket Description,sentiment_label,sentiment_score
0,390,"I'm having an issue with the Canon EOS. Please assist. 0/4 - 0.1 (0.00%) and 1.0 (0.00%) will receive a refund once their order is placed I've tried using different cables, adapters, or peripherals with my Canon EOS, but the issue persists.",NEGATIVE,0.994046
1,822,"I'm having an issue with the Lenovo ThinkPad. Please assist. Please, if possible, buy the . Please, if possible, sell the . If you can't, I've performed a factory reset on my Lenovo ThinkPad, hoping it would resolve the problem, but it didn't help.",NEGATIVE,0.998216
2,1292,"I'm having an issue with the iPhone. Please assist. Thank you. The product_purchased contains all our products and our services are the exclusive property of our supplier, our distributors, and our respective affiliates I'm concerned about the security of my iPhone and would like to ensure that my data is safe.",POSITIVE,0.997281
3,1419,"I'm having an issue with the Nikon D. Please assist. Thank you. Please keep everything updated. Update 5: I received an email from an editor who was using that same device. She said she wanted I've checked for any available software updates for my Nikon D, but there are none.",POSITIVE,0.833919
4,1438,"I'm having an issue with the Bose SoundLink Speaker. Please assist. I was unable to download the Product ID for all the new ones that were added on September 20. You can visit this website for any product you find that you need The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.",NEGATIVE,0.990343
5,1902,I'm having an issue with the Google Pixel. Please assist. [08:01:29] [Client thread/INFO]: [CHAT] [Server: I want the package ID to be 'C:\Program Files I need assistance as soon as possible because it's affecting my work and productivity.,NEGATIVE,0.998023
6,2160,I'm having an issue with the Google Pixel. Please assist. Thank you for your support. The PLEx (PLEx) app is an Android application for your iOS devices. It combines features of PLE I'm concerned about the security of my Google Pixel and would like to ensure that my data is safe.,POSITIVE,0.991469
7,2387,"I'm having an issue with the iPhone. Please assist. I could use a few more people for the project... Thank you! --@Brick Brick is an open source software, and we are currently working I've recently updated the firmware of my iPhone, and the issue started happening afterward. Could it be related to the update?",POSITIVE,0.974268
8,2508,"The Amazon Kindle is unable to establish a stable internet connection. It keeps disconnecting intermittently. How can I troubleshoot this network problem? This is easy. [00:15:20]ACCESS: I've followed online tutorials and community forums to troubleshoot the issue, but no luck so far.",NEGATIVE,0.999378
9,4016,"I'm having an issue with the Google Nest. Please assist. 6) If I use your product, will the seller use this. 7) If the seller doesn't sell it, will the buyer use this I've recently updated the firmware of my Google Nest, and the issue started happening afterward. Could it be related to the update?",NEGATIVE,0.998134


## 7. Manual Validation

A targeted manual review is used to assess whether model predictions align with the underlying meaning of selected customer support tickets.

A ticket is manually labeled as `NEGATIVE` when it primarily expresses a problem, failure, concern, urgency, or unresolved difficulty. A ticket is labeled as `POSITIVE` only when it expresses a genuinely favorable or satisfactory experience.

The validation sample is balanced across predicted sentiment classes to provide diagnostic coverage of both labels. Therefore, the resulting agreement rate is treated as a qualitative model check rather than an estimate of overall model accuracy.

In [65]:
# Record manual sentiment labels based on the primary meaning of each ticket.
manual_labels = [
    "NEGATIVE",  # 390
    "NEGATIVE",  # 822
    "NEGATIVE",  # 1292
    "NEGATIVE",  # 1419
    "NEGATIVE",  # 1438
    "NEGATIVE",  # 1902
    "NEGATIVE",  # 2160
    "NEGATIVE",  # 2387
    "NEGATIVE",  # 2508
    "NEGATIVE",  # 4016
    "NEGATIVE",  # 4233
    "NEGATIVE",  # 4297
    "NEGATIVE",  # 4756
    "NEGATIVE",  # 6513
    "NEGATIVE",  # 7094
    "NEGATIVE",  # 7534
    "NEGATIVE",  # 7716
    "NEGATIVE",  # 7756
    "NEGATIVE",  # 8387
    "NEGATIVE",  # 8390
]

validation_sample["manual_sentiment"] = manual_labels

# Compare manual assessments with the model's predicted sentiment.
validation_sample["agreement"] = (
    validation_sample["sentiment_label"]
    == validation_sample["manual_sentiment"]
)

validation_sample[
    [
        "Ticket ID",
        "sentiment_label",
        "manual_sentiment",
        "sentiment_score",
        "agreement",
    ]
]

,Ticket ID,sentiment_label,manual_sentiment,sentiment_score,agreement
0,390,NEGATIVE,NEGATIVE,0.994046,True
1,822,NEGATIVE,NEGATIVE,0.998216,True
2,1292,POSITIVE,NEGATIVE,0.997281,False
3,1419,POSITIVE,NEGATIVE,0.833919,False
4,1438,NEGATIVE,NEGATIVE,0.990343,True
5,1902,NEGATIVE,NEGATIVE,0.998023,True
6,2160,POSITIVE,NEGATIVE,0.991469,False
7,2387,POSITIVE,NEGATIVE,0.974268,False
8,2508,NEGATIVE,NEGATIVE,0.999378,True
9,4016,NEGATIVE,NEGATIVE,0.998134,True


In [66]:
# Calculate agreement between manual review labels and model predictions.
agreement_rate = validation_sample["agreement"].mean() * 100

print(
    f"Agreement rate: "
    f"{validation_sample['agreement'].sum()} / {len(validation_sample)} "
    f"({agreement_rate:.1f}%)"
)

Agreement rate: 10 / 20 (50.0%)


## 8. Observed Model Limitations

Manual review of the validation sample identified limitations in applying a general-purpose sentiment model to customer support tickets.

### Surface-level positive language

Some tickets describing problems were classified as `POSITIVE` when they contained phrases such as "Thank you" or other polite language. The model may respond to these surface-level signals rather than the overall meaning of the support issue.

### Problem statements classified as positive

Several tickets clearly described unresolved issues, including security concerns, firmware problems, intermittent failures, and difficulty completing an action, but were predicted as `POSITIVE`.

### Noisy and incoherent text

Some ticket descriptions contain unrelated, malformed, or contextually inconsistent text. This noise can introduce sentiment signals that do not reflect the customer's actual support issue.

### Domain mismatch

The pretrained model was trained for general English sentiment classification rather than customer support communication. As a result, model confidence should not be interpreted as a direct measure of correctness for this domain.

The targeted validation sample achieved 10 agreements out of 20 reviewed tickets (50.0%). Because the sample was balanced by predicted sentiment class, this result is used as a diagnostic indicator rather than an estimate of overall model accuracy.

## 9. Export Sentiment Results

The final sentiment output contains one prediction for each ticket.

The exported dataset includes:

- `ticket_id`
- `sentiment_label`
- `sentiment_score`

This output will be used for later analytical integration.

In [67]:
# Prepare the final ticket-level sentiment output for downstream integration.
sentiment_output = sentiment_results.rename(
    columns={"Ticket ID": "ticket_id"}
)

sentiment_output.head()

,ticket_id,sentiment_label,sentiment_score
0,1,NEGATIVE,0.994507
1,2,NEGATIVE,0.948731
2,3,NEGATIVE,0.999365
3,4,NEGATIVE,0.967981
4,5,NEGATIVE,0.997396


In [68]:
# Confirm the final output has the expected structure and complete ticket coverage.
print(f"Output shape: {sentiment_output.shape}")
print(f"Unique ticket IDs: {sentiment_output['ticket_id'].nunique()}")
print(f"Missing sentiment labels: {sentiment_output['sentiment_label'].isna().sum()}")
print(f"Missing sentiment scores: {sentiment_output['sentiment_score'].isna().sum()}")

Output shape: (8469, 3)
Unique ticket IDs: 8469
Missing sentiment labels: 0
Missing sentiment scores: 0


In [69]:
# Define the output location for the ticket-level sentiment predictions.
SENTIMENT_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "ticket_sentiment.csv"
)

# Export the sentiment predictions without the DataFrame index.
sentiment_output.to_csv(
    SENTIMENT_OUTPUT_PATH,
    index=False,
)

print(f"Sentiment results exported to: {SENTIMENT_OUTPUT_PATH}")

Sentiment results exported to: c:\Users\roshi\DA_Sprint\projects\ai-customer-support-analytics\data\processed\ticket_sentiment.csv


In [70]:
# Verify the exported file can be read successfully and matches the expected structure.
exported_sentiment = pd.read_csv(SENTIMENT_OUTPUT_PATH)

print(f"Exported shape: {exported_sentiment.shape}")
print(f"Duplicate ticket IDs: {exported_sentiment['ticket_id'].duplicated().sum()}")

display(exported_sentiment.head())

Exported shape: (8469, 3)
Duplicate ticket IDs: 0


,ticket_id,sentiment_label,sentiment_score
0,1,NEGATIVE,0.994507
1,2,NEGATIVE,0.948731
2,3,NEGATIVE,0.999365
3,4,NEGATIVE,0.967981
4,5,NEGATIVE,0.997396
